In [ ]:
try:
    from google.colab import drive  # type: ignore
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    !cp /content/drive/MyDrive/BuildGPT/Part5/name.txt /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn.functional as F
import random
from typing import Any

In [ ]:
with open("name.txt", "r") as file:
    words = file.read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [ ]:
chars: list[str] = sorted(set("".join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

vocab_size = len(stoi)

In [ ]:
random.seed(42)
random.shuffle(words)

In [ ]:
# block_size = 3
block_size = 8

def build_data_set(words: list[str]):
    X, Y = [], []
    for word in words:
        context = [0] * block_size
        for char in word + ".":
            ix = stoi[char]
            X.append(context)  # X 存储 context 字符索引
            Y.append(ix)  # Y 存储预测 的字符索引
            context = context[1:] + [ix]  # 每次都把上下文的第一个字符索引去掉

    return torch.tensor(X), torch.tensor(Y)


X, Y = build_data_set(words)
print(f"X shape: {X.shape}, Y shape:{Y.shape}")

n_words = len(words)
n1 = int(0.8 * n_words)
n2 = int(0.9 * n_words)

X_train, Y_train = build_data_set(words[:n1])
X_dev, Y_dev = build_data_set(words[n1:n2])
X_test, Y_test = build_data_set(words[n2:])

X shape: torch.Size([228146, 8]), Y shape:torch.Size([228146])


In [ ]:
class Linear:
    # fan_out: 神经元的数量
    # fan_in:单个神经元可以接受多少个参数
    def __init__(self, fan_in: int, fan_out: int, bias=True):
        self.weight = torch.randn(fan_in, fan_out) * fan_in ** (-0.5)  # kaiming init
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x: torch.Tensor):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

In [ ]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1) -> None:
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # batch 均值和方差
        self.x_mean = torch.zeros(dim)
        self.x_var = torch.zeros(dim)

        # 全局均值和方差
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.zeros(dim)

    def __call__(self, x: torch.Tensor) -> Any:
        if x.dim() == 2:
            dims = (0,)
        elif x.dim() == 3:
            dims = (0, 1)
        else:
            dims = (0,)

        if self.training:
            x_mean = x.mean(dim=dims, keepdim=True)
            x_var = x.var(dim=dims, keepdim=True)
        else:
            x_mean = self.running_mean
            x_var = self.running_var

        x_hat = (x - x_mean) / ((x_var + self.eps) ** (0.5))  # batch normalization
        self.out = self.gamma * x_hat + self.beta

        # train 的过程同时计算全局 mean, var
        if self.training:
            with torch.no_grad():
                self.running_mean = (
                    1 - self.momentum
                ) * self.running_mean + self.momentum * x_mean
                self.running_var = (
                    1 - self.momentum
                ) * self.running_var + self.momentum * x_var

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

**三维 Tensor 的平均**

```python
T.type : torch.Tensor
T.shape = (3, 5, 4)
```

在这个矩阵中，一共有 3 × 5 = 15 个格子。

每一个格子里的元素 $v_{i,j}$ 不是一个单独的数值，而是一个包含 4 个分量的向量：$$v_{i,j} = [c_1, c_2, c_3, c_4]$$

其中 $i$ 表示行号（1 到 3），$j$ 表示列号（1 到 5）。

执行 mean(dim=(0, 1))

15 个c_1 求平均, 15 个c_2 求平均, 15 个c_3 求平均, 15 个c_4 求平均, 最终的 Tensor.shape = (1, 1, 4)

---

pytorch 原生的 BatchNorm1d 要求的dim 的输入是 (N, C, L)

N, batch; C, Features or Channels; L, Sequence Length

假设我们当前输出的张量是 (32, 8, 200)（32 个样本，8 个单词，200 个特征）。

如果我们直接把它扔给 PyTorch 原生的 torch.nn.BatchNorm1d(200)：

PyTorch 会生搬硬套 (N, C, L) 的模子。

它会把我们的 $T=8$ 误认为通道数 $C$。

它会把我们的 $C=200$ 误认为时间长度 $L$。

结果：由于它初始化时准备了 200 个通道的 $\gamma$ 和 $\beta$，但却发现你的“通道”只有 8 个，程序会直接报错崩溃。

```python
# 标准的做法是在前后各加一次“维度置换（Transpose）”
x = x.transpose(1, 2)    # 把 (B, T, C) 扭曲成 (B, C, T) 来迎合 PyTorch
x = native_batch_norm(x) # 此时满足 (N, C, L) 格式，正常计算
x = x.transpose(1, 2)    # 算完再扭曲回 (B, T, C)
```
好消息是现在几乎不会使用 BatchNorm1d 了, 后面会转向层归一化

In [ ]:
class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

In [ ]:
class Embedding:
    # num_embbding, 要嵌入到多少个元素中
    # embedding_dim, 每个元素需要多少个特征
    def __init__(self, num_embbding, embedding_dim) -> None:
        self.weight = torch.randn(num_embbding, embedding_dim)

    def __call__(self, IX):  # IX 表示这里嵌入的是 indices, 区别与其它 x
        self.out = self.weight[IX]
        return self.out

    def parameters(self):
        return [self.weight]

In [ ]:
class Flatten:
    def __call__(self, x):
        self.out = x.view(x.shape[0], -1)
        return self.out

    def parameters(self):
        return []

In [ ]:
class FlattenConsecutive:
    def __init__(self, n) -> None:
        self.n = n

    def __call__(self, x:torch.Tensor):
        B = x.shape[0] # batch
        T = x.shape[1] # Time
        C = x.shape[2] # Channel
        # // 表示整除, 向下取整
        # 必须重新赋值给 x, view 并不会直接改变调用的 x
        x = x.view(B, T // self.n, C * self.n)

        if x.shape[1] == 1: # 如果该维度被 flatten 到 1 了, 那就 取消该维度, 也就是视为 2 维
          # 必须重新赋值给 x, squeeze 并不会直接改变调用的 x
            x = x.squeeze(1)

        self.out = x

        return self.out

    def parameters(self):
        return []

***Flatten 的本质***

假设一个输入张量，形状为 $(32, 8, 10)$, $32$ 个句子, 每个句子有 $8$ 个单词, 每个单词用长度为 $10$ 。

使用 FlattenConsecutive 并且设定每次融合相邻的 $2$ 个单词：它会把第 1、2 个单词的向量拼接，第 3、4 个拼接，依此类推。

单词数维度被压缩一半（$8 \div 2 = 4$），而最后的向量维度(描述单个单词的向量的分量)膨胀一倍（$10 \times 2 = 20$）。

结果形状：$(32, 4, 20)$。

网络像“锦标赛”或者“树状图”一样，一层一层地把低级特征（单字）两两组合成高级特征（词组）

---


***切片（Slicing）***

Python 的切片语法完整公式是：
sequence[start:stop:step]

这三个参数分别代表：

start (起始位置)：从哪个索引开始切（包含该位置）。如果不写，默认从头（索引 0）开始。

stop (结束位置)：切到哪个索引为止（不包含该位置）。如果不写，默认一直切到序列的最后。

step (步长)：每次取值的跨度。如果不写，默认跨度是 1（挨个取）。

当你看到两个冒号 :: 连在一起时，意味着中间的 stop 参数被省略了。

___


***拼接(Concatenates)***

```python
原始数据
A = [
      [ [1, 2, 3],[4, 5, 6] ],
      [ [7, 8, 9],[10, 11, 12] ]
    ]  # 形状 (2, 2, 3)

B = [
      [ [-1, -2, -3],[-4, -5, -6] ],
      [ [-7, -8, -9],[-10, -11, -12] ]
    ]  # 形状 (2, 2, 3)


torch.cat(A,B, dim = 0) # 行拼接
结果:(B 拼接到 A 的下方)
[
  [ [1, 2, 3],[4, 5, 6] ],
  [ [7, 8, 9],[10, 11, 12] ],
  [ [-1, -2, -3],[-4, -5, -6] ],
  [ [-7, -8, -9],[-10, -11, -12] ]
]  # 形状 (4, 2, 3)

torch.cat(A,B, dim = 1) # 列拼接
结果:(B 拼接到 A 的右侧)
[
  [ [ [1, 2, 3],[4, 5, 6], [-1, -2, -3],[-4, -5, -6] ],
  [ [7, 8, 9],[10, 11, 12], [-7, -8, -9],[-10, -11, -12] ]
]# 形状 (2, 4, 3)

torch.cat(A,B, dim = 2) # 元素分量拼接
[
      [ [1, 2, 3, -1, -2, -3],[4, 5, 6, -4, -5, -6] ],
      [ [7, 8, 9,-7, -8, -9],[10, 11, 12, -10, -11, -12] ]
]  # 形状 (2, 2, 6)
```

___

***切片 + 拼接***

```python
e = torch.randn((4,4,8))

# e[:, 1::2, :] 的效果是第 0,2 维全部都取, 第1 维从1开始, 按步长2 取
explicit = torch.cat(e[:, ::2, :], e[:, 1::2, :]. dim = 2)

e[:, ::2, :] 看作 A
e[:, 1::2, :] 看作 B
然后拼接, 原则同前
```

In [ ]:
from typing import Any


class Sequential:
    def __init__(self, layers: list[Any]) -> None:
        self.layers = layers

    def __call__(self, x: torch.Tensor) -> Any:
        for layer in self.layers:
            x = layer(x)

        self.out = x

        return self.out

    def parameters(self) -> list[torch.Tensor]:
        return [p for layer in self.layers for p in layer.parameters()]  # 列表推导

In [ ]:
torch.manual_seed(42)

**最初的形式, 下面会在此基础上优化**
```python
# 初始化
embdding_dim = 10
n_hidden = 200


layers = [
    Embedding(vocab_size, embdding_dim),
    Flatten(),
    Linear(embdding_dim * block_size, n_hidden, bias=False),
    BatchNorm1d(n_hidden), # x线性层的列, 会自动行扩展
    Tanh(), Linear(n_hidden, vocab_size)]

with torch.no_grad():
    # 模型初始状态下应该是无知的,平等的看待任何元素
    # 因而人为的调整 logits 为 0, 这里仅接近 0, 没有设置为绝对 0
    layers[-1].weight *= 0.1

parameters:list[torch.Tensor] = [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True
```

___

```python
max_steps = 20000
batch_size = 32

lossi = []
for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,))
    Xb, Yb = X_train[ix], Y_train[ix] # train 嵌入到 ix
    
    # forward pass
    x= Xb
    for layer in layers:
        x=layer(x)
    # x 即 logits
    loss = F.cross_entropy(x, Yb)
    
    lr = 0.1 if i < 150000 else 0.01

    for p in parameters:
        if p.grad is not None:
            p.data += -lr * p.grad

    print(f"{i:7d} / {max_steps:7d}: {loss.item():.4f}")
    
    lossi.append(loss.log10().item())
    
    break
```

***v1***
```python
# 初始化
embdding_dim = 10
n_hidden = 200


model = Sequential(
    [
        Embedding(vocab_size, embdding_dim),
        Flatten(),
        Linear(embdding_dim * block_size, n_hidden, bias=False),
        BatchNorm1d(n_hidden),  # x线性层的列, 会自动行扩展
        Tanh(),
        Linear(n_hidden, vocab_size),
    ]
)

with torch.no_grad():
    # 模型初始状态下应该是无知的,平等的看待任何元素
    # 因而人为的调整 logits 为 0, 这里仅接近 0, 没有设置为绝对 0
    model.layers[-1].weight *= 0.1

parameters = model.parameters()
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True
```

In [ ]:
# 初始化
embdding_dim = 10
n_hidden = 200
flatten_size = 2

model = Sequential(
    [
        Embedding(vocab_size, embdding_dim),

        FlattenConsecutive(flatten_size),
        # 为什么是 embdding_dim * flatten_size, 参见下方 @ 运算机制解释
        Linear(embdding_dim * flatten_size, n_hidden, bias=False),
        BatchNorm1d(n_hidden),  # x线性层的列, 会自动行扩展
        Tanh(),

        FlattenConsecutive(flatten_size),
        Linear(n_hidden * flatten_size, n_hidden, bias=False),
        BatchNorm1d(n_hidden),  # x线性层的列, 会自动行扩展
        Tanh(),

        FlattenConsecutive(flatten_size),
        Linear(n_hidden * flatten_size, n_hidden, bias=False),
        BatchNorm1d(n_hidden),  # x线性层的列, 会自动行扩展
        Tanh(),

        Linear(n_hidden, vocab_size),
    ]
)

with torch.no_grad():
    # 模型初始状态下应该是无知的,平等的看待任何元素
    # 因而人为的调整 logits 为 0, 这里仅接近 0, 没有设置为绝对 0
    model.layers[-1].weight *= 0.1

parameters = model.parameters()
print(sum(p.nelement() for p in parameters))

for p in parameters:
    p.requires_grad = True

170897


**pytorch 的 "@" 运算机制**

@ 被官方定义为矩阵乘法运算符。在 PyTorch 中，A @ B 完全等价于调用 `torch.matmul(A, B)`

输入张量 $X$：形状是 (4, 5, 6, 80)

权重矩阵 $W$：形状是 (80, 200)

对于$X$,  4 * 5 * 6 = 120, 4, 5, 6 虚的, 只有最底层的 80 存储了实际的数据

在运算时 $W$ 会克隆为 120 份(逻辑克隆, 或者说仅仅是1分数据参与120次运算, 并没有开辟新的内存)

每一份都与 $X$ 做运算, 过程大致为

```c++
for(i = 0; i < 4; ++i){
  for(j = 0; j < 5; ++j){
    for(k = 0; k < 6; ++k){
          X[i,j,k] //取出 80 个数
    }
  }
}

for(b = 0; b < 120; ++b){ // 120 是虚的
  for(c = 0; c < 200; ++c){
        W[b,c]//取出 80 个数, b 是虚的, 实际不存在这个维度, 仅仅为了解释
  }
}

```
$$
\begin{align}
& \hspace{100cm} \\
& X[i,j,k] = {x_{ijk1}, x_{ijk2}, ..., x_{ijk80}} \\
& W[b,c] = {w_{bc1}, w_{bc2}, ..., w_{bc80}} \\
& out_1 = x_{ijk1} * w_{b1,1}, x_{ijk2} * w_{b1,2}, ..., x_{ijk80} * w_{b1,80} \\
& out_2 = x_{ijk1} * w_{b21}, x_{ijk2} * w_{b2,2}, ..., x_{ijk80} * w_{b2,80} \\
& \vdots \\
& out_{200} = x_{ijk1} * w_{b200,1}, x_{ijk2} * w_{b200,2}, ..., x_{ijk80} * w_{b200,80} \\
& 如果不考虑 W 虚的维度, 合理的计算式应为 \\
& out_{i,j,k,c} = \sum_{m=1}^{80} X_{i,j,k,m} \cdot W_{m,c}\\
\end{align}
$$

120 组 200 个 $out$ 会替换掉 $X$ 原来的4 * 5 * 6 组 80 个数据

---
**Gemini 的解释**

PyTorch 的 @ (matmul) 高维运算机制

当一个 N 维张量 $X$ 遇到一个 2 维权重矩阵 $W$ 时：

1. 维度拆分：
- 地址维 (Batch Dims)：$X$ 前面的所有维度 $(4, 5, 6)$。
- 数据维 (Feature Dim)：$X$ 的最后一个维度 $(80)$，必须与 $W$ 的行数匹配。

2. 核心逻辑：
- $W$ 像是一个通用的加工厂 $(80 \rightarrow 200)$。
- PyTorch 会自动“穿透”前面的地址维，在每一个独立的工位 $(i, j, k)$ 上，将那里的 $80$ 个数据与 $W$ 的 $200$ 个神经元分别做点积。

3. 结果替换：
- 每个工位上的 $80$ 个原始分量被加工后的 $200$ 个新分量原地替换。
- 形状变化：$(4, 5, 6, 80) \xrightarrow{@ (80, 200)} (4, 5, 6, 200)$。

一句话总结：
前面的维度决定了“在哪算”（保持不变），最后的维度决定了“算什么”（发生变换）。


In [ ]:
ix_temp = torch.randint(0,X_train.shape[0], (4,))
X_tr_b, Y_tr_b = X_train[ix_temp], Y_train[ix_temp]
logits_temp = model(X_tr_b)

# 要在 logits = model(Xb) 执行之后才会有有 out
for layer in model.layers:
  print(f"{type(layer).__name__} shape: {layer.out.shape}")

Embedding shape: torch.Size([4, 8, 10])
FlattenConsecutive shape: torch.Size([4, 4, 20])
Linear shape: torch.Size([4, 4, 200])
BatchNorm1d shape: torch.Size([4, 4, 200])
Tanh shape: torch.Size([4, 4, 200])
FlattenConsecutive shape: torch.Size([4, 2, 400])
Linear shape: torch.Size([4, 2, 200])
BatchNorm1d shape: torch.Size([4, 2, 200])
Tanh shape: torch.Size([4, 2, 200])
FlattenConsecutive shape: torch.Size([4, 400])
Linear shape: torch.Size([4, 200])
BatchNorm1d shape: torch.Size([4, 200])
Tanh shape: torch.Size([4, 200])
Linear shape: torch.Size([4, 27])


1. 魔术方法 (Magic Methods)：
这部分和 C++ 的运算符重载一模一样。这些是以双下划线包围的函数。它们的魔术在于“你定义了它，但你几乎永远不会直接调用它的名字”，

 __add__(self, other) $\rightarrow$ 用的时候写 a + b

__call__(self, *args) $\rightarrow$ 用的时候写 obj()写

__bool__(self) $\rightarrow$ 你用的时候写 if obj:

__len__(self) $\rightarrow$ 你用的时候写 len(obj)

这里的魔术是行为映射（Behavior Mapping）。

2. 魔术属性 (Magic Attributes)：无中生有的魔术

__class__、__name__、__dict__ 这些是变量（属性），不是函数。

魔术之一：由解释器“无中生有”地注入。

在你写 class Linear: 时，你并没有在 __init__ 里写 self.__class__ = Linear，也没有写 self.__name__ = 'Linear'。

这些属性是 Python 底层的元类（Metaclass）在构建这个类时，悄悄塞进内存里的。

魔术之二：打破次元壁，直通 C 语言底层。

在 CPython（Python 的官方 C 实现）的源码中，每一个 Python 对象在底层都是一个 C 语言结构（PyObject）。

__class__ 这个属性不是一个普通的字典键值对，它实际上是一根指针，直接指向了底层 C 结构体里的 ob_type 字段。

魔术之三：保留字与命名空间隔离。

Python 发明者 Guido 规定了双下划线作为系统保留的命名空间。开发者写业务代码定义变量时，千万别起名叫 __xxx__，以免破坏系统的魔法阵。

In [ ]:
######################## train ####################################
max_steps = 200000
batch_size = 32

lossi = []
for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,))
    Xb, Yb = X_train[ix], Y_train[ix]  # train 嵌入到 ix

    # forward pass
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)

    # zero gradients
    for p in parameters:
      p.grad = None # 避免梯度累加

    # backward pass
    loss.backward()

    # update parameters
    lr = 0.1 if i < 150000 else 0.01

    for p in parameters:
        if p.grad is not None:
            p.data += -lr * p.grad

    # log
    if i % 10000 == 0:
      print(f"{i:7d} / {max_steps:7d}: {loss.item():.4f}")

    lossi.append(loss.log10().item())

    #break

      0 /  200000: 3.2970
  10000 /  200000: 2.0723
  20000 /  200000: 2.0520
  30000 /  200000: 2.1174
  40000 /  200000: 2.1276
  50000 /  200000: 2.5542
  60000 /  200000: 2.2019
  70000 /  200000: 2.0805
  80000 /  200000: 2.2914
  90000 /  200000: 1.7958
 100000 /  200000: 1.6737
 110000 /  200000: 1.5885
 120000 /  200000: 1.7018
 130000 /  200000: 1.7855
 140000 /  200000: 1.7009
 150000 /  200000: 1.9260
 160000 /  200000: 1.8774
 170000 /  200000: 1.8569
 180000 /  200000: 1.7865
 190000 /  200000: 1.7791


In [ ]:
######################## evaluate the loss ####################################
@torch.no_grad()
def split_loss(split: str):
    x, y = {
        "train": (X_train, Y_train),
        "dev": (X_dev, Y_dev),
        "test": (X_test, Y_test),
    }[split] # 从 map 中取出

    logits = model(x)
    loss = F.cross_entropy(logits, y)

    print(f"{split}: {loss.item():.4f}")

split_loss("train")
split_loss("dev")

train: 1.7326
dev: 1.9912


`block_size = 3`:

train: 2.0578

dev: 2.1055

---

`block_size = 8`:

train: 1.9161

dev: 2.0304


仅仅是 block_size 的提升, 就可以将模型质量由 2.1055 提升至 2.0304

---

由全 flatten 结构调整为渐进式融合的分层结构, loss 由2.03 降低至1.99, 有一定的提升, 但不大

train: 1.7326

dev: 1.9912

In [ ]:
######################## sample from the model ####################################
for layer in model.layers:
    layer.training = False # 解释见下


for _ in range(20):
    context = [0] * block_size
    out=[]
    while(True):
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item() # 采样结果是下标, 优先采样概率(权重)高的元素
        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0: # 预测的是 "."
            break
    print(''.join(itos[i] for i in out))


**为什么要 layer.training = False**

`class BatchNorm1d` 中，默认状态是 `self.training = True`

意味着哪怕只有 1 个样本，它也要强行去计算当前 Batch 的方差。

统计学中计算无偏样本方差（Bessel's correction）的公式，需要除以 $N - 1$：

$$s^2 = \frac{1}{N-1} \sum_{i=1}^{N} (x_i - \bar{x})^2$$

当 $N = 1$（只有一个样本）时，分母变成了 $0$！PyTorch 计算单样本方差会直接返回 nan。

归一化代码 `(x - x_mean) / sqrt(x_var + eps)` 因为分母含有 nan，输出全变成了 nan。